# ChiNeutro: ajuste de mT5 / mT0 (encoder-decoder) en Colab T4

Ajusta un modelo encoder-decoder para traducir español de Chile a español neutro con el **split oficial del grupo**
(`data/entrenamiento/splits/split_oficial.csv`: 3.184 train / 399 val / 397 test) y deja las predicciones en el
formato de `evaluacion/` (`id_mistral, frase_chilena, prediccion`).

**Cómo usarlo**
1. Entorno de ejecución → Cambiar tipo → **T4 GPU**.
2. Ajusta la celda de configuración (`MODELO_ID`, `MODO`, `LORA_R`...).
3. Ejecutar todo. Los checkpoints y predicciones se guardan en Drive (`MyDrive/chineutro_mt5/<NOMBRE>/`). Si Colab se
   desconecta, vuelve a ejecutar todo y el entrenamiento retoma desde el último checkpoint.

**Decisiones**
- **Mismo prompt que la baseline de mT0** (`INSTRUCCION_CHAT` de `models/gemma4/comun.py`), así mT0 sin ajuste vs
  ajustado es una comparación justa.
- **fp32 en T4**: mT5/mT0 dan NaN en fp16 y la T4 no tiene bf16 nativo. En A100 se usa bf16 automáticamente.
- **Generación greedy y primera línea**, igual que la baseline.
- **val** para comparar configuraciones. **test** solo para el resultado final (con juez, en tu computador).

In [ ]:
%pip install -q -U "transformers>=4.56" "peft>=0.13" accelerate sacrebleu bert-score==0.3.13
%pip install -q sentencepiece "protobuf<7"
%pip uninstall -q -y torchao   # el torchao 0.10 de Colab rompe a peft al crear el LoRA; no se usa

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución -> Cambiar tipo -> T4 GPU"
p = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.get_device_capability(0)[0] >= 8      # A100/L4 sí, T4 no
print(f"GPU: {p.name} | {p.total_memory / 1e9:.0f} GB | bf16 nativo: {BF16}")

## Configuración

In [ ]:
# Modelo de partida
MODELO_ID = "bigscience/mt0-large"   # "bigscience/mt0-large" | "google/mt5-large" | "google/mt5-base"
MODO = "lora"                        # "lora" (recomendado en T4) | "full" (solo con mt5-base en T4)

# LoRA: mismos valores que el paso 2 de Gemma (r16, alpha 32) para poder comparar
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
# Atención (q, k, v, o) y MLP (wi_0, wi_1, wo) del encoder y del decoder, como "atencion+mlp" en Gemma
LORA_MODULOS = ["q", "k", "v", "o", "wi_0", "wi_1", "wo"]

# Entrenamiento
LR = 5e-4 if MODO == "lora" else 3e-4   # T5 tolera LR más altos que los LLM
EPOCAS = 6                              # se queda con la mejor época según la pérdida de validación
LOTE = 16                               # si da OOM: LOTE = 8 y ACUM = 2
ACUM = 1
SEED = 42

MAX_LEN_IN, MAX_LEN_OUT, MAX_NEW_TOKENS = 128, 96, 96

# Repo
REPO_URL = "https://github.com/matiaraos/NLP-ChiNeutro.git"
RAMA = "baseline"                       # la que tiene el split oficial y evaluacion/

corto = MODELO_ID.split("/")[-1].replace("-", "_")
NOMBRE = f"{corto}_{MODO}_r{LORA_R}" if MODO == "lora" else f"{corto}_full"
print("Corrida:", NOMBRE)

## Drive y repo

In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DIR_DRIVE = Path("/content/drive/MyDrive/chineutro_mt5") / NOMBRE
DIR_CKPT = DIR_DRIVE / "checkpoints"
DIR_PRED = DIR_DRIVE / "predicciones"
for d in (DIR_CKPT, DIR_PRED):
    d.mkdir(parents=True, exist_ok=True)

# Si el repo es privado: agrega un secreto de Colab llamado GITHUB_TOKEN (llave a la izquierda) con un token de
# GitHub con permiso de lectura. Si es público, no hace falta.
RAIZ = Path("/content/NLP-ChiNeutro")
if not RAIZ.exists():
    url = REPO_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    r = subprocess.run(["git", "clone", "-q", "-b", RAMA, url, str(RAIZ)], capture_output=True, text=True)
    assert r.returncode == 0, "No pude clonar el repo (¿privado? agrega el secreto GITHUB_TOKEN)"
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 1. Datos (split oficial)

In [ ]:
import pandas as pd
from evaluacion import cargar_split

por_split = {s: cargar_split(s) for s in ("train", "val", "test")}
tamanos = {s: len(d) for s, d in por_split.items()}
print(tamanos)
assert tamanos == {"train": 3184, "val": 399, "test": 397}, "El split no es el oficial"

# Ninguna frase chilena puede repetirse entre splits
frases = {s: set(d.frase_chilena.str.strip().str.lower()) for s, d in por_split.items()}
assert not (frases["train"] & frases["val"]) and not (frases["train"] & frases["test"]) \
    and not (frases["val"] & frases["test"]), "fuga entre splits"

# Mismo prompt que la baseline de mT0 (models/gemma4/comun.py)
sys.path.insert(0, str(RAIZ / "models/gemma4"))
from comun import INSTRUCCION_CHAT
def entrada(frase):
    return INSTRUCCION_CHAT.format(chilena=frase.strip())
print(entrada(por_split["train"].frase_chilena[0]))

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(MODELO_ID)

def a_dataset(df):
    x = tok([entrada(f) for f in df.frase_chilena], max_length=MAX_LEN_IN, truncation=True)
    y = tok(text_target=[n.strip() for n in df.frase_neutralizada], max_length=MAX_LEN_OUT, truncation=True)
    return Dataset.from_dict({"input_ids": x["input_ids"], "attention_mask": x["attention_mask"],
                              "labels": y["input_ids"]})

ds_train, ds_val = a_dataset(por_split["train"]), a_dataset(por_split["val"])
largo_in = max(len(i) for i in ds_train["input_ids"])
largo_out = max(len(l) for l in ds_train["labels"])
print(f"tokens máx: entrada {largo_in}, salida {largo_out}")
assert largo_in < MAX_LEN_IN and largo_out < MAX_LEN_OUT, "algún ejemplo se truncó: sube MAX_LEN_IN/OUT"

## 2. Modelo

In [ ]:
from transformers import AutoModelForSeq2SeqLM, set_seed

set_seed(SEED)
dtype = torch.bfloat16 if BF16 else torch.float32       # nunca fp16 con mT5/mT0
modelo = AutoModelForSeq2SeqLM.from_pretrained(MODELO_ID, dtype=dtype)
modelo.config.use_cache = False

if MODO == "lora":
    from peft import LoraConfig, get_peft_model
    modelo = get_peft_model(modelo, LoraConfig(task_type="SEQ_2_SEQ_LM", r=LORA_R, lora_alpha=LORA_ALPHA,
                                               lora_dropout=LORA_DROPOUT, target_modules=LORA_MODULOS))
    modelo.print_trainable_parameters()
else:
    print(f"Parámetros entrenables: {sum(p.numel() for p in modelo.parameters()) / 1e6:.0f} M")

## 3. Entrenamiento

Evalúa la pérdida en val al final de cada época y se queda con la mejor. Checkpoints en Drive: si Colab se corta,
vuelve a ejecutar todo y retoma.

In [ ]:
import math
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

pasos_totales = math.ceil(len(ds_train) / (LOTE * ACUM)) * EPOCAS
args = Seq2SeqTrainingArguments(
    output_dir=str(DIR_CKPT),
    per_device_train_batch_size=LOTE,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACUM,
    num_train_epochs=EPOCAS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=int(0.05 * pasos_totales),
    optim="adamw_torch" if MODO == "lora" else "adafactor",
    bf16=BF16,
    fp16=False,
    gradient_checkpointing=(MODO == "full"),
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
)
trainer = Seq2SeqTrainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                         data_collator=DataCollatorForSeq2Seq(tok, label_pad_token_id=-100))

hay_ckpt = any(DIR_CKPT.glob("checkpoint-*"))
print("Retomando desde checkpoint" if hay_ckpt else "Entrenamiento desde cero")
trainer.train(resume_from_checkpoint=hay_ckpt)
trainer.save_model(str(DIR_DRIVE / "modelo_final"))
tok.save_pretrained(str(DIR_DRIVE / "modelo_final"))

In [ ]:
import math, json
import matplotlib.pyplot as plt

hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
assert not any(math.isnan(l) for _, l in tr + ev), "La pérdida dio NaN: revisa que no esté en fp16"

plt.figure(figsize=(7, 3.5))
plt.plot(*zip(*tr), label="train")
plt.plot(*zip(*ev), "o-", label="val")
plt.xlabel("paso"); plt.ylabel("pérdida"); plt.title(NOMBRE, loc="left"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(DIR_DRIVE / "curva_perdida.png", dpi=150); plt.show()
print("Mejor checkpoint:", trainer.state.best_model_checkpoint, "| eval_loss:", trainer.state.best_metric)
json.dump(hist, open(DIR_DRIVE / "historial.json", "w"), indent=1)

## 4. Predicciones (val y test)

In [ ]:
modelo = trainer.model.eval()
modelo.config.use_cache = True

def traducir(frases, bs=32):
    salidas = []
    for i in range(0, len(frases), bs):
        x = tok([entrada(f) for f in frases[i:i + bs]], return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_LEN_IN).to(modelo.device)
        with torch.no_grad():
            out = modelo.generate(input_ids=x["input_ids"], attention_mask=x["attention_mask"],
                                  do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS)
        # igual que la baseline: primera línea
        salidas += [s.strip().split("\n")[0].strip() for s in tok.batch_decode(out, skip_special_tokens=True)]
    return salidas

RUTAS = {}
for s in ("val", "test"):
    df = por_split[s][["id_mistral", "frase_chilena"]].copy()
    df["prediccion"] = traducir(df.frase_chilena.tolist())
    RUTAS[s] = DIR_PRED / f"predicciones_{NOMBRE}_{s}.csv"
    df.to_csv(RUTAS[s], index=False)
    print(f"{s}: {RUTAS[s].name} | vacías: {(df.prediccion == '').sum()} "
          f"| copia literal: {(df.prediccion.str.strip() == df.frase_chilena.str.strip()).mean():.1%}")

In [ ]:
# Ejemplos de val para revisar a ojo
ej = por_split["val"][["termino", "frase_chilena", "frase_neutralizada"]].copy()
ej["prediccion"] = pd.read_csv(RUTAS["val"]).prediccion
pd.set_option("display.max_colwidth", 120)
display(ej.sample(12, random_state=SEED))

## 5. Métricas automáticas en val (sin juez)

BLEU, % de términos chilenos retenidos y BERTScore con el pipeline del grupo. Úsalas para comparar configuraciones.
El juez (Opus vía Claude Code) se corre después en tu computador, sobre test, con la configuración ya elegida.

In [ ]:
import gc
del trainer, modelo
gc.collect(); torch.cuda.empty_cache()

from evaluacion import evaluar
resumen, _ = evaluar(str(RUTAS["val"]), juez=False)
print(json.dumps(resumen, ensure_ascii=False, indent=2))